In [1]:
import json
 
# Load stored data
simple_wiki_data_path = "simple_wiki_raw_data.json"

with open(simple_wiki_data_path,"r") as file:
    simple_wiki_data = json.load(file)

len(simple_wiki_data), simple_wiki_data[0]

(751,
 {'url': 'https://simple.wikipedia.org/wiki/Software',
  'title': 'Software',
  'sections': [{'heading': 'Introduction',
    'paragraphs': ['Computer software, also called software, is a set of instructions and documentation that tells a computer what to do or how to perform a task. Software includes all different programs on a computer, such as applications and the operating system. Applications are programs that are designed to perform a specific operation, such as a game or a word processor. The operating system (e.g., macOS, Microsoft Windows, Android and various Linux distributions) is a type of software that is used as a platform for running the applications, and controls all user interface tools including display and the keyboard.',
     'The word software was first used in the late 1960s to emphasize on its difference from computer hardware, which can be physically observed by the user. Software is a set of instructions that the computer follows. Before compact discs (CDs

In [2]:
page_titles = [page.get('title') for page in simple_wiki_data]
simple_page_urls = [page.get('url') for page in simple_wiki_data]
len(page_titles), page_titles[:5]

(751,
 ['Software',
  'Kernel (computer science)',
  'Regression toward the mean',
  'Analytics',
  'Autoregressive model'])

In [3]:
# (try to) convert simple wiki page url to its classic wiki page url counterpart

def simple2normalwiki_url(simple_url):
    page_url_segment = simple_url.split("/")[-1]
    normalwiki_base = "https://en.wikipedia.org/wiki/"
    normalwiki_url = normalwiki_base + page_url_segment
    return normalwiki_url

test_simple_url = "https://simple.wikipedia.org/wiki/Numerical_integration"
test_normal_url = simple2normalwiki_url(test_simple_url)
test_normal_url

'https://en.wikipedia.org/wiki/Numerical_integration'

In [4]:
from bs4 import BeautifulSoup, Tag
import requests
import re


# ----------------------------------------
# 			  SCRAPING METHOD
# ----------------------------------------

In [5]:
# ---------------- CONFIG ----------------
IGNORE_CLASSES = {
    "sidebar-list", "navbar", "infobox", "toc",
    "thumb", "mw-default-size", "metadata"
}

STOP_SECTIONS = {
    "references", "external links", "see also", "notes", "further reading"
}


In [6]:
# ---------------- HELPERS ----------------
def clean_paragraph(el: Tag) -> str:
    """Clean paragraph text, preserving math as LaTeX."""

    # Remove citation markers
    for sup in el.find_all("sup"):
        sup.decompose()

    # Preserve math
    for math in el.find_all("math"):
        latex = math.get("alttext") or math.get_text(strip=True)
        latex = latex.strip()

        is_block = el.get_text(strip=True) == math.get_text(strip=True)
        math.replace_with(
            f"\n$$\n{latex}\n$$\n" if is_block else f"${latex}$"
        )

    # Lists
    if el.name in {"ul", "ol"}:
        lines = []
        for i, li in enumerate(el.find_all("li", recursive=False), start=1):
            txt = clean_paragraph(li)
            if txt:
                lines.append(f"- {txt}" if el.name == "ul" else f"{i}) {txt}")
        return "\n".join(lines)

    # Text cleanup
    text = el.get_text(" ", strip=True)
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"\s+([.,;:!?])", r"\1", text)
    return text.strip()


def is_ignored(el: Tag) -> bool:
    """Ignore elements inside navboxes, infoboxes, thumbnails, TOC."""
    for parent in el.parents:
        classes = parent.get("class", [])
        if any(cls in IGNORE_CLASSES for cls in classes):
            return True
    return False


In [7]:
# ---------------- MAIN SCRAPER ----------------
def scrape_normal_wiki(url: str) -> dict:
    headers = {"User-Agent": "ReverseMentorBot/0.1"}
    res = requests.get(url, headers=headers)

    if res.status_code != 200:
        return {
            "url": url,
            "error": "page does not exist - no (normal) wiki page counterpart to this simple wiki page"
        }

    soup = BeautifulSoup(res.text, "html.parser")

    # ✅ Correct container (works for Software, modern pages, old pages)
    content = soup.find("div", id="mw-content-text")
    if not content:
        return {
            "url": url,
            "error": "content not found"
        }

    sections = []
    intro = None
    current = None

    # Traverse in DOM order
    for el in content.find_all(
        ["p", "li", "dd", "ul", "ol", "h2", "h3", "h4", "h5"],
        recursive=True
    ):
        if is_ignored(el):
            continue

        # ---------- HEADINGS ----------
        if el.name.startswith("h"):
            heading = el.get_text(" ", strip=True).replace("[edit]", "")
            if heading.lower() in STOP_SECTIONS:
                break

            current = {"heading": heading, "paragraphs": []}
            sections.append(current)
            continue

        # ---------- CONTENT ----------
        text = clean_paragraph(el)
        if not text:
            continue

        if current is None:
            if intro is None:
                intro = {"heading": "Introduction", "paragraphs": []}
                sections.insert(0, intro)
            intro["paragraphs"].append(text)
        else:
            current["paragraphs"].append(text)

    title_tag = soup.find("h1")
    title = title_tag.get_text(strip=True) if title_tag else None

    return {
        "url": url,
        "title": title,
        "sections": sections
    }


In [8]:
# --- Test scraper ---

test_normal_url1 = 'https://en.wikipedia.org/wiki/Numerical_integra'
test1 = scrape_normal_wiki(test_normal_url1)
print(test1)

test_normal_url2 = 'https://en.wikipedia.org/wiki/Poisson_distribution'
test2 = scrape_normal_wiki(test_normal_url2)
test2


{'url': 'https://en.wikipedia.org/wiki/Numerical_integra', 'error': 'page does not exist - no (normal) wiki page counterpart to this simple wiki page'}


{'url': 'https://en.wikipedia.org/wiki/Poisson_distribution',
 'title': 'Poisson distribution',
 'sections': [{'heading': 'Introduction',
   'paragraphs': ['In probability theory and statistics, the Poisson distribution ( / ˈ p w ɑː s ɒ n / ) is a discrete probability distribution that expresses the probability of a given number of events occurring in a fixed interval of time if these events occur with a known constant mean rate and independently of the time since the last event. It can also be used for the number of events in other types of intervals than time, and in dimension greater than 1 (e.g., number of events in a given area or volume). The Poisson distribution is named after French mathematician Siméon Denis Poisson. It plays an important role for discrete-stable distributions.',
    'Under a Poisson distribution with the expectation of λ events in a given interval, the probability of k events in the same interval is: ${\\displaystyle {\\frac {\\lambda ^{k}e^{-\\lambda }}{k!}}

# add storing function from simple wiki notebook -> helpers.py

In [ ]:
from concurrent.futures import ThreadPoolExecutor
from data_helpers import load_cache, save_cache, merge_article_into_cache
import time



# --------------------------------------------------------
# SCRAPING NORMAL WIKI PAGES FROM SIMPLE WIKI PAGES STORED
# --------------------------------------------------------

CACHE_FILE = "normal_wiki_raw_data.json"

def get_stored_simple_wiki():
    simple_wiki_data_path = "simple_wiki_raw_data.json"
    with open(simple_wiki_data_path,"r") as file:
            simple_wiki_data = json.load(file)
    return simple_wiki_data


# (try to) convert simple wiki page url to its classic wiki page url counterpart
def simple2normalwiki_url(simple_url):
    page_url_segment = simple_url.split("/")[-1]
    normalwiki_base = "https://en.wikipedia.org/wiki/"
    normalwiki_url = normalwiki_base + page_url_segment
    return normalwiki_url


def scrape_all_normal(max_workers=10, cache_file=None): 
    # get all stored simple wiki data
    simple_wiki_data = get_stored_simple_wiki()
    simple_urls = [page['url'] for page in simple_wiki_data]
    
	# (try to) convert simple wiki urls to normal wiki urls
    normal_urls = [simple2normalwiki_url(url) for url in simple_urls]
    
    t1 = time.perf_counter()
    
	# scrape normal wiki pages in parallel using normal wiki page urls
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
            scraped_pages = list(executor.map(scrape_normal_wiki, normal_urls))
            t2 = time.perf_counter()

            if cache_file:
                cache = load_cache(cache_file)
                for page in scraped_pages:
                        merge_article_into_cache(
                            cache,
                            page,
                            issimple=False,
                            )
                save_cache(cache, cache_file)
            t3 = time.perf_counter()
            
    print(f"scraping time: {t2 - t1:.2f}s")
    print(f"storing time: {t3 - t2:.2f}s")
                    
    return scraped_pages


normal_wiki_data = scrape_all_normal(max_workers=10, cache_file=CACHE_FILE)

print(f"nb of scraped pages (+potentially stored): {len(normal_wiki_data)}")
normal_wiki_data 

In [10]:
# To add individual normal wiki page (not from simple wiki page store)

CACHE_FILE = "normal_wiki_raw_data.json"

def scrape_individual(url, CACHE_FILE=None):
    
	indiv_page = scrape_normal_wiki(url=url)
	if CACHE_FILE:
		with open("normal_wiki_raw_data.json","r") as file:
			stored_normal_wiki_data = json.load(file)

		issimple = 'simple.wikipedia.org' in url

		merge_article_into_cache(cache=stored_normal_wiki_data,
						   article_dict=indiv_page,
						   issimple=issimple)
		
		save_cache(cache=stored_normal_wiki_data, CACHE_FILE=CACHE_FILE)
	
	return indiv_page

tony = scrape_individual(url='https://en.wikipedia.org/wiki/Tony_Stark_(Marvel_Cinematic_Universe)', CACHE_FILE=CACHE_FILE)
rob = scrape_individual('https://en.wikipedia.org/wiki/Robert_Downey_Jr.', CACHE_FILE=CACHE_FILE)


# verify if new page stored correctly by doing lookup in the cache
with open('normal_wiki_raw_data.json', 'r') as file:
	stored_normal_wiki_data = json.load(file)

print(f'size of cache: {len(stored_normal_wiki_data)} pages')

lookup_titles = ['Tony Stark', 'Robert Downey Jr']

[page 
 for page in stored_normal_wiki_data 
 if any(title in page.get('title', '') for title in lookup_titles)]

size of cache: 753 pages


[{'url': 'https://en.wikipedia.org/wiki/Tony_Stark_(Marvel_Cinematic_Universe)',
  'title': 'Tony Stark (Marvel Cinematic Universe)',
  'sections': [{'heading': 'Introduction',
    'paragraphs': ['Anthony Edward Stark is a fictional character primarily portrayed by Robert Downey Jr. in the Marvel Cinematic Universe (MCU) media franchise —based on the Marvel Comics character of the same name —commonly known by his alias, Iron Man. Stark is initially depicted as an industrialist, genius inventor, and former playboy who is CEO of Stark Industries. Initially the chief weapons manufacturer for the U.S. military, he has a change of heart and redirects his technical knowledge into creating mechanized suits of armor, which he uses to defend Earth.',
     "Stark becomes a founding member and eventual leader of the Avengers. Following his failed Ultron Program, the internal conflict within the Avengers due to the Sokovia Accords, and Thanos successfully erasing half of all life in the Blip, Star

In [11]:
with open("normal_wiki_raw_data.json","r") as file:
    stored_normal_wiki_data = json.load(file)

print(f"nb of stored pages: {len(stored_normal_wiki_data)}")

[page for page in stored_normal_wiki_data if 'Robert Downey' in page.get('title', '')]

nb of stored pages: 753


[{'url': 'https://en.wikipedia.org/wiki/Robert_Downey_Jr.',
  'title': 'Robert Downey Jr.',
  'sections': [{'heading': 'Introduction',
    'paragraphs': ["Robert John Downey Jr. (born April 4, 1965) is an American actor. Known for portraying charismatic and intelligent characters over a diverse range of films, he was the highest-paid actor in Hollywood annually from 2013 to 2015. His films as a leading actor have grossed over $14.3 billion worldwide, making him one of the highest-grossing actors of all time. Downey's accolades include an Academy Award, a Daytime Emmy Award, three Golden Globe Awards, and two BAFTA Awards.",
     "At the age of five, Downey made his acting debut in his father Robert Downey Sr. 's film Pound (1970). He rose to prominence by working with the Brat Pack for the teen films Weird Science (1985) and Less than Zero (1987). His portrayal of Charlie Chaplin in the biopic Chaplin (1992) earned him the BAFTA Award for Best Actor and an Academy Award nomination. Aft

In [12]:
# Display page url that do not exist (inexistant mapping from simple wiki page url)

content_not_found = [
    page for page in stored_normal_wiki_data
    if 'error' in page
]

print(f"nb inexistant mapping from simple wiki pages: {len(content_not_found)}")
content_not_found

nb inexistant mapping from simple wiki pages: 21


[{'url': 'https://en.wikipedia.org/wiki/Inference_(statistics)',
  'error': 'page does not exist - no (normal) wiki page counterpart to this simple wiki page',
  'last_scraped': '2026-02-03T14:36:23.754214+00:00',
  'first_scraped': '2026-02-03T08:04:57.477189+00:00'},
 {'url': 'https://en.wikipedia.org/wiki/AI-generated_erotic_art',
  'error': 'page does not exist - no (normal) wiki page counterpart to this simple wiki page',
  'last_scraped': '2026-02-03T14:36:23.755260+00:00',
  'first_scraped': '2026-02-03T08:04:57.477542+00:00'},
 {'url': 'https://en.wikipedia.org/wiki/CopilotAI_(Microsoft)',
  'error': 'page does not exist - no (normal) wiki page counterpart to this simple wiki page',
  'last_scraped': '2026-02-03T14:36:23.755412+00:00',
  'first_scraped': '2026-02-03T08:04:57.477594+00:00'},
 {'url': 'https://en.wikipedia.org/wiki/Decision_stream_graph',
  'error': 'page does not exist - no (normal) wiki page counterpart to this simple wiki page',
  'last_scraped': '2026-02-03T1

In [13]:
# from IPython.display import display, Markdown
from display_helpers import pretty_print_page
from random import choice

# # --- Pretty Print Functionality ---
# def pretty_print_sections(data):
#     md = f"# {data['title']}\n\n"
    
#     for section in data['sections']:
#         md += f"## {section['heading']}\n\n"
#         for para in section["paragraphs"]:
#             md += para + "\n\n"
#     md += "\n------------------------------------------------------------------"
#     display(Markdown(md))

rand_normal = choice(stored_normal_wiki_data)
pretty_print_page(rand_normal)

# Monte Carlo algorithm

## Introduction

In computing, a Monte Carlo algorithm is a randomized algorithm whose output may be incorrect with a certain (typically small) probability. Two examples of such algorithms are the Karger–Stein algorithm and the Monte Carlo algorithm for minimum feedback arc set.

The name refers to the Monte Carlo casino in the Principality of Monaco, which is well-known around the world as an icon of gambling. The term "Monte Carlo" was first introduced in 1947 by Nicholas Metropolis.

Las Vegas algorithms are a dual of Monte Carlo algorithms and never return an incorrect answer. However, they may make random choices as part of their work. As a result, the time taken might vary between runs, even with the same input.

If there is a procedure for verifying whether the answer given by a Monte Carlo algorithm is correct, and the probability of a correct answer is bounded above zero, then with probability one, running the algorithm repeatedly while testing the answers will eventually give a correct answer. Whether this process is a Las Vegas algorithm depends on whether halting with probability one is considered to satisfy the definition.

## One-sided vs two-sided error

While the answer returned by a deterministic algorithm is always expected to be correct, this is not the case for Monte Carlo algorithms. For decision problems, these algorithms are generally classified as either false -biased or true -biased. A false -biased Monte Carlo algorithm is always correct when it returns false; a true -biased algorithm is always correct when it returns true. While this describes algorithms with one-sided errors, others might have no bias; these are said to have two-sided errors. The answer they provide (either true or false ) will be incorrect, or correct, with some bounded probability.

For instance, the Solovay–Strassen primality test is used to determine whether a given number is a prime number. It always answers true for prime number inputs; for composite inputs, it answers false with probability at least 1 ⁄ 2 and true with probability less than 1 ⁄ 2. Thus, false answers from the algorithm are certain to be correct, whereas the true answers remain uncertain; this is said to be a 1 ⁄ 2 -correct false-biased algorithm.

## Amplification

For a Monte Carlo algorithm with one-sided errors, the failure probability can be reduced (and the success probability amplified) by running the algorithm k times. Consider again the Solovay–Strassen algorithm which is 1 ⁄ 2 -correct false-biased. One may run this algorithm multiple times returning a false answer if it reaches a false response within k iterations, and otherwise returning true. Thus, if the number is prime then the answer is always correct, and if the number is composite then the answer is correct with probability at least 1−(1− 1 ⁄ 2 ) = 1−2.

For Monte Carlo decision algorithms with two-sided error, the failure probability may again be reduced by running the algorithm k times and returning the majority function of the answers.

## Complexity classes

The complexity class BPP describes decision problems that can be solved by polynomial-time Monte Carlo algorithms with a bounded probability of two-sided errors, and the complexity class RP describes problems that can be solved by a Monte Carlo algorithm with a bounded probability of one-sided error: if the correct answer is false, the algorithm always says so, but it may answer false incorrectly for some instances where the correct answer is true. In contrast, the complexity class ZPP describes problems solvable by polynomial expected time Las Vegas algorithms. ZPP ⊆ RP ⊆ BPP, but it is not known whether any of these complexity classes is distinct from each other; that is, Monte Carlo algorithms may have more computational power than Las Vegas algorithms, but this has not been proven. Another complexity class, PP, describes decision problems with a polynomial-time Monte Carlo algorithm that is more accurate than flipping a coin but where the error probability cannot necessarily be bounded away from 1 ⁄ 2.

## Classes of Monte Carlo and Las Vegas algorithms

Randomized algorithms are primarily divided by its two main types, Monte Carlo and Las Vegas, however, these represent only a top of the hierarchy and can be further categorized.

- Las Vegas Sherwood—"performant and effective special case of Las Vegas" Numerical —"numerical Las Vegas"
- Monte Carlo Atlantic City —"bounded error special case of Monte Carlo" Numerical—"numerical approximation Monte Carlo"

Las Vegas Sherwood—"performant and effective special case of Las Vegas" Numerical —"numerical Las Vegas"

- Sherwood—"performant and effective special case of Las Vegas"
- Numerical —"numerical Las Vegas"

Sherwood—"performant and effective special case of Las Vegas"

Numerical —"numerical Las Vegas"

Monte Carlo Atlantic City —"bounded error special case of Monte Carlo" Numerical—"numerical approximation Monte Carlo"

- Atlantic City —"bounded error special case of Monte Carlo"
- Numerical—"numerical approximation Monte Carlo"

Atlantic City —"bounded error special case of Monte Carlo"

Numerical—"numerical approximation Monte Carlo"

"Both Las Vegas and Monte Carlo are dealing with decisions, i.e., problems in their decision version." "This however should not give a wrong impression and confine these algorithms to such problems—both types of randomized algorithms can be used on numerical problems as well, problems where the output is not simple ‘yes’/‘no’, but where one needs to receive a result that is numerical in nature."

(stronger bound than regular LV)

Sherwood probabilistic

will inhibit usefulness of the algorithm; typical case is ${\displaystyle <{\tfrac {1}{2}}}$ )

Previous table represents a general framework for Monte Carlo and Las Vegas randomized algorithms. Instead of the mathematical symbol ${\displaystyle <}$ one could use ${\displaystyle \leq }$, thus making probabilities in the worst case equal.

## Applications in computational number theory and other areas

Well-known Monte Carlo algorithms include the Solovay–Strassen primality test, the Baillie–PSW primality test, the Miller–Rabin primality test, and certain fast variants of the Schreier–Sims algorithm in computational group theory.

For algorithms that are a part of Stochastic Optimization (SO) group of algorithms, where probability is not known in advance and is empirically determined, it is sometimes possible to merge Monte Carlo and such an algorithm "to have both probability bound calculated in advance and a Stochastic Optimization component." "Example of such an algorithm is Ant Inspired Monte Carlo." In this way, "drawback of SO has been mitigated, and a confidence in a solution has been established."


------------------------------------------------------------------

In [17]:
from random import choices

# display multiple pages
rand_selection = choices(stored_normal_wiki_data, k=2)
for page in rand_selection:
    pretty_print_page(page)

# Confusion and diffusion

## Introduction

In cryptography, confusion and diffusion are two properties of a secure cipher identified by Claude Shannon in his 1945 classified report A Mathematical Theory of Cryptography. These properties, when present, work together to thwart the application of statistics, and other methods of cryptanalysis.

Confusion in a symmetric cipher is obscuring the local correlation between the input ( plaintext ), and output ( ciphertext ) by varying the application of the key to the data, while diffusion is hiding the plaintext statistics by spreading it over a larger area of ciphertext. Although ciphers can be confusion-only ( substitution cipher, one-time pad ) or diffusion-only ( transposition cipher ), any "reasonable" block cipher uses both confusion and diffusion. These concepts are also important in the design of cryptographic hash functions, and pseudorandom number generators, where decorrelation of the generated values is the main feature. Diffusion (and its avalanche effect ) is also applicable to non-cryptographic hash functions.

## Definition

## Confusion

Confusion means that each binary digit (bit) of the ciphertext should depend on several parts of the key, obscuring the connections between the two.

The property of confusion hides the relationship between the ciphertext and the key.

This property makes it difficult to find the key from the ciphertext and if a single bit in a key is changed, the calculation of most or all of the bits in the ciphertext will be affected.

Confusion increases the ambiguity of ciphertext and it is used by both block and stream ciphers.

In substitution–permutation networks, confusion is provided by substitution boxes.

## Diffusion

Diffusion means that if we change a single bit of the plaintext, then about half of the bits in the ciphertext should change, and similarly, if we change one bit of the ciphertext, then about half of the plaintext bits should change. This is equivalent to the expectation that encryption schemes exhibit an avalanche effect.

The purpose of diffusion is to hide the statistical relationship between the ciphertext and the plain text. For example, diffusion ensures that any patterns in the plaintext, such as redundant bits, are not apparent in the ciphertext. Block ciphers achieve this by "diffusing" the information about the plaintext's structure across the rows and columns of the cipher.

In substitution–permutation networks, diffusion is provided by permutation boxes (a.k.a. permutation layer ). In the beginning of the 21st century a consensus had appeared where the designers preferred the permutation layer to consist of linear Boolean functions, although nonlinear functions can be used, too.

## Theory

In Shannon's original definitions, confusion refers to making the relationship between the ciphertext and the symmetric key as complex and involved as possible; diffusion refers to dissipating the statistical structure of plaintext over the bulk of ciphertext. This complexity is generally implemented through a well-defined and repeatable series of substitutions and permutations. Substitution refers to the replacement of certain components (usually bits) with other components, following certain rules. Permutation refers to manipulation of the order of bits according to some algorithm. To be effective, any non-uniformity of plaintext bits needs to be redistributed across much larger structures in the ciphertext, making that non-uniformity much harder to detect.

In particular, for a randomly chosen input, if one flips the i -th bit, then the probability that the j -th output bit will change should be one half, for any i and j —this is termed the strict avalanche criterion. More generally, one may require that flipping a fixed set of bits should change each output bit with probability one half.

One aim of confusion is to make it very hard to find the key even if one has a large number of plaintext-ciphertext pairs produced with the same key. Therefore, each bit of the ciphertext should depend on the entire key, and in different ways on different bits of the key. In particular, changing one bit of the key should change the ciphertext completely.

## Practical applications

Design of a modern block cipher uses both confusion and diffusion, with confusion changing data between the input and the output by applying a key-dependent non-linear transformation (linear calculations are easier to reverse and thus are easier to break).

Confusion inevitably involves some diffusion, so a design with a very wide-input S-box can provide the necessary diffusion properties, but will be very costly in implementation. Therefore, the practical ciphers utilize relatively small S-boxes, operating on small groups of bits ("bundles" ). For example, the design of AES has 8-bit S-boxes, Serpent − 4-bit, BaseKing and 3-way − 3-bit. Small S-boxes provide almost no diffusion, so the resources are spent on simpler diffusion transformations. For example, the wide trail strategy popularized by the Rijndael design, involves a linear mixing transformation that provides high diffusion, although the security proofs do not depend on the diffusion layer being linear.

One of the most researched cipher structures uses the substitution-permutation network (SPN) where each round includes a layer of local nonlinear permutations ( S-boxes ) for confusion and a linear diffusion transformation (usually a multiplication by a matrix over a finite field ). Modern block ciphers mostly follow the confusion layer/diffusion layer model, with the efficiency of the diffusion layer estimated using the so-called branch number, a numerical parameter that can reach the value ${\displaystyle s+1}$ for s input bundles for the perfect diffusion transformation. Since the transformations that have high branch numbers (and thus require a lot of bundles as inputs) are costly in implementation, the diffusion layer is sometimes (for example, in the AES) composed from two sublayers, "local diffusion" that processes subsets of the bundles in a bricklayer fashion (each subset is transformed independently) and "dispersion" that makes the bits that were "close" (within one subset of bundles) to become "distant" (spread to different subsets and thus be locally diffused within these new subsets on the next round).

## Analysis of AES

The Advanced Encryption Standard (AES) has both excellent confusion and diffusion. Its confusion look-up tables are very non-linear and good at destroying patterns. Its diffusion stage spreads every part of the input to every part of the output: changing one bit of input changes half the output bits on average. Both confusion and diffusion are repeated multiple times for each input to increase the amount of scrambling. The secret key is mixed in at every stage so that an attacker cannot precalculate what the cipher does.

None of this happens when a simple one-stage scramble is based on a key. Input patterns would flow straight through to the output. It might look random to the eye but analysis would find obvious patterns and the cipher could be broken.


------------------------------------------------------------------

# Euler's totient function

## Introduction

In number theory, Euler's totient function counts the positive integers up to a given integer ${\displaystyle n}$ that are relatively prime to ${\displaystyle n}$. It is written using the Greek letter phi as ${\displaystyle \varphi (n)}$ or ${\displaystyle \phi (n)}$, and may also be called Euler's phi function. In other words, it is the number of integers ${\displaystyle k}$ in the range ${\displaystyle 1\leq k\leq n}$ for which the greatest common divisor ${\displaystyle \gcd(n,k)}$ is equal to 1. The integers ${\displaystyle k}$ of this form are sometimes referred to as totatives of ${\displaystyle n}$.

For example, the totatives of ${\displaystyle n=9}$ are the six numbers 1, 2, 4, 5, 7 and 8. They are all relatively prime to 9, but the other three numbers in this range, 3, 6, and 9 are not, since ${\displaystyle \gcd(9,3)=\gcd(9,6)=3}$ and ${\displaystyle \gcd(9,9)=9}$. Therefore, ${\displaystyle \varphi (9)=6}$. As another example, ${\displaystyle \varphi (1)=1}$ since for ${\displaystyle n=1}$ the only integer in the range from 1 to ${\displaystyle n}$ is 1 itself, and ${\displaystyle \gcd(1,1)=1}$.

Euler's totient function is a multiplicative function, meaning that if two numbers ${\displaystyle m}$ and ${\displaystyle n}$ are relatively prime, then ${\displaystyle \varphi (mn)=\varphi (m)\varphi (n)}$. This function gives the order of the multiplicative group of integers modulo n (the group of units of the ring ${\displaystyle \mathbb {Z} /n\mathbb {Z} }$ ). It is also used for defining the RSA encryption system.

## History, terminology, and notation

Leonhard Euler introduced the function in 1763. However, he did not at that time choose any specific symbol to denote it. In a 1784 publication, Euler studied the function further, choosing the Greek letter ${\displaystyle \pi }$ to denote it: he wrote ${\displaystyle \pi D}$ for "the multitude of numbers less than ${\displaystyle D}$, and which have no common divisor with it". This definition varies from the current definition for the totient function at ${\displaystyle D=1}$ but is otherwise the same. The now-standard notation ${\displaystyle \varphi (A)}$ comes from Gauss 's 1801 treatise Disquisitiones Arithmeticae, although Gauss did not use parentheses around the argument and wrote ${\displaystyle \varphi A}$. Thus, it is often called Euler's phi function or simply the phi function.

In 1879, J. J. Sylvester coined the term totient for this function, so it is also referred to as Euler's totient function, the Euler totient, or Euler's totient. Jordan's totient is a generalization of Euler's.

The cototient of ${\displaystyle n}$ is defined as ${\displaystyle n-\varphi (n)}$. It counts the number of positive integers less than or equal to ${\displaystyle n}$ that have at least one prime factor in common with ${\displaystyle n}$.

## Computing Euler's totient function

There are several formulae for computing ${\displaystyle \varphi (n)}$.

## Euler's product formula

It states

$$ {\displaystyle \varphi (n)=n\prod _{p\mid n}\left(1-{\frac {1}{p}}\right),} $$

where the product is over the distinct prime numbers dividing n.

An equivalent formulation is

$$ {\displaystyle \varphi (n)=p_{1}^{k_{1}-1}(p_{1}{-}1)\,p_{2}^{k_{2}-1}(p_{2}{-}1)\cdots p_{r}^{k_{r}-1}(p_{r}{-}1),} $$

where ${\displaystyle n=p_{1}^{k_{1}}p_{2}^{k_{2}}\cdots p_{r}^{k_{r}}}$ is the prime factorization of ${\displaystyle n}$ (that is, ${\displaystyle p_{1},p_{2},\ldots,p_{r}}$ are distinct prime numbers).

The proof of these formulae depends on two important facts.

## Phi is a multiplicative function

This means that if ${\displaystyle \gcd(m,n)=1}$, then ${\displaystyle \varphi (m)\varphi (n)=\varphi (mn)}$. Proof outline: Let ${\displaystyle A,B,C}$ be the sets of positive integers which are coprime to and less than m, n, mn, respectively, so that ${\displaystyle |A|=\varphi (m)}$, etc. Then there is a bijection between ${\displaystyle A\times B}$ and C by the Chinese remainder theorem.

## Value of phi for a prime power argument

If p is prime and ${\displaystyle k\geq 1}$, then

$$ {\displaystyle \varphi \left(p^{k}\right)=p^{k}-p^{k-1}=p^{k-1}(p-1)=p^{k}\left(1-{\tfrac {1}{p}}\right).} $$

Proof: Since p is a prime number, the only possible values of ${\displaystyle \gcd(p^{k},m)}$ are ${\displaystyle 1,p,p^{2},\dots,p^{k}}$, and the only way to have ${\displaystyle \gcd(p^{k},m)>1}$ is if m is a multiple of p, that is, ${\displaystyle m\in \{p,2p,3p,\ldots,p^{k-1}p=p^{k}\}}$, and there are ${\displaystyle p^{k-1}}$ such multiples not greater than ${\displaystyle p^{k}}$. Therefore, the other ${\displaystyle p^{k}-p^{k-1}}$ numbers are all relatively prime to ${\displaystyle p^{k}}$.

## Proof of Euler's product formula

The fundamental theorem of arithmetic states that if n > 1 there is a unique expression ${\displaystyle n=p_{1}^{k_{1}}p_{2}^{k_{2}}\cdots p_{r}^{k_{r}},}$ where p 1 < p 2 <... < p r are prime numbers and each k i ≥ 1. (The case n = 1 corresponds to the empty product.) Repeatedly using the multiplicative property of φ and the formula for φ ( p ) gives

$$ {\displaystyle {\begin{array}{rcl}\varphi (n)&=&\varphi (p_{1}^{k_{1}})\,\varphi (p_{2}^{k_{2}})\cdots \varphi (p_{r}^{k_{r}})\\[.1em]&=&p_{1}^{k_{1}}\left(1-{\frac {1}{p_{1}}}\right)p_{2}^{k_{2}}\left(1-{\frac {1}{p_{2}}}\right)\cdots p_{r}^{k_{r}}\left(1-{\frac {1}{p_{r}}}\right)\\[.1em]&=&p_{1}^{k_{1}}p_{2}^{k_{2}}\cdots p_{r}^{k_{r}}\left(1-{\frac {1}{p_{1}}}\right)\left(1-{\frac {1}{p_{2}}}\right)\cdots \left(1-{\frac {1}{p_{r}}}\right)\\[.1em]&=&n\left(1-{\frac {1}{p_{1}}}\right)\left(1-{\frac {1}{p_{2}}}\right)\cdots \left(1-{\frac {1}{p_{r}}}\right).\end{array}}} $$

This gives both versions of Euler's product formula.

An alternative proof that does not require the multiplicative property instead uses the inclusion-exclusion principle applied to the set ${\displaystyle \{1,2,\ldots,n\}}$, excluding the sets of integers divisible by the prime divisors.

## Example

$$ {\displaystyle \varphi (20)=\varphi (2^{2}5)=20\,(1-{\tfrac {1}{2}})\,(1-{\tfrac {1}{5}})=20\cdot {\tfrac {1}{2}}\cdot {\tfrac {4}{5}}=8.} $$

In words: the distinct prime factors of 20 are 2 and 5; half of the twenty integers from 1 to 20 are divisible by 2, leaving ten; a fifth of those are divisible by 5, leaving eight numbers coprime to 20; these are: 1, 3, 7, 9, 11, 13, 17, 19.

The alternative formula uses only integers: ${\displaystyle \varphi (20)=\varphi (2^{2}5^{1})=2^{2-1}(2{-}1)\,5^{1-1}(5{-}1)=2\cdot 1\cdot 1\cdot 4=8.}$

## Fourier transform

The totient is the discrete Fourier transform of the gcd, evaluated at 1. Let

$$ {\displaystyle {\mathcal {F}}\{\mathbf {x} \}[m]=\sum \limits _{k=1}^{n}x_{k}\cdot e^{{-2\pi i}{\frac {mk}{n}}}} $$

where x k = gcd( k, n ) for k ∈ {1,..., n }. Then

$$ {\displaystyle \varphi (n)={\mathcal {F}}\{\mathbf {x} \}[1]=\sum \limits _{k=1}^{n}\gcd(k,n)e^{-2\pi i{\frac {k}{n}}}.} $$

The real part of this formula is

$$ {\displaystyle \varphi (n)=\sum \limits _{k=1}^{n}\gcd(k,n)\cos {\tfrac {2\pi k}{n}}.} $$

For example, using ${\displaystyle \cos {\tfrac {\pi }{5}}={\tfrac {{\sqrt {5}}+1}{4}}}$ and ${\displaystyle \cos {\tfrac {2\pi }{5}}={\tfrac {{\sqrt {5}}-1}{4}}}$: ${\displaystyle {\begin{array}{rcl}\varphi (10)&=&\gcd(1,10)\cos {\tfrac {2\pi }{10}}+\gcd(2,10)\cos {\tfrac {4\pi }{10}}+\gcd(3,10)\cos {\tfrac {6\pi }{10}}+\cdots +\gcd(10,10)\cos {\tfrac {20\pi }{10}}\\&=&1\cdot ({\tfrac {{\sqrt {5}}+1}{4}})+2\cdot ({\tfrac {{\sqrt {5}}-1}{4}})+1\cdot (-{\tfrac {{\sqrt {5}}-1}{4}})+2\cdot (-{\tfrac {{\sqrt {5}}+1}{4}})+5\cdot (-1)\\&&+\ 2\cdot (-{\tfrac {{\sqrt {5}}+1}{4}})+1\cdot (-{\tfrac {{\sqrt {5}}-1}{4}})+2\cdot ({\tfrac {{\sqrt {5}}-1}{4}})+1\cdot ({\tfrac {{\sqrt {5}}+1}{4}})+10\cdot (1)\\&=&4.\end{array}}}$ Unlike the Euler product and the divisor sum formula, this one does not require knowing the factors of n. However, it does involve the calculation of the greatest common divisor of n and every positive integer less than n, which suffices to provide the factorization anyway.

## Divisor sum

The property established by Gauss, that

$$ {\displaystyle \sum _{d\mid n}\varphi (d)=n,} $$

where the sum is over all positive divisors d of n, can be proven in several ways. (See Arithmetical function for notational conventions.)

One proof is to note that φ ( d ) is also equal to the number of possible generators of the cyclic group C d; specifically, if C d = ⟨ g ⟩ with g = 1, then g is a generator for every k coprime to d. Since every element of C n generates a cyclic subgroup, and each subgroup C d ⊆ C n is generated by precisely φ ( d ) elements of C n, the formula follows. Equivalently, the formula can be derived by the same argument applied to the multiplicative group of the n th roots of unity and the primitive d th roots of unity.

The formula can also be derived from elementary arithmetic. For example, let n = 20 and consider the positive fractions up to 1 with denominator 20:

$$ {\displaystyle {\tfrac {1}{20}},\,{\tfrac {2}{20}},\,{\tfrac {3}{20}},\,{\tfrac {4}{20}},\,{\tfrac {5}{20}},\,{\tfrac {6}{20}},\,{\tfrac {7}{20}},\,{\tfrac {8}{20}},\,{\tfrac {9}{20}},\,{\tfrac {10}{20}},\,{\tfrac {11}{20}},\,{\tfrac {12}{20}},\,{\tfrac {13}{20}},\,{\tfrac {14}{20}},\,{\tfrac {15}{20}},\,{\tfrac {16}{20}},\,{\tfrac {17}{20}},\,{\tfrac {18}{20}},\,{\tfrac {19}{20}},\,{\tfrac {20}{20}}.} $$

Put them into lowest terms:

$$ {\displaystyle {\tfrac {1}{20}},\,{\tfrac {1}{10}},\,{\tfrac {3}{20}},\,{\tfrac {1}{5}},\,{\tfrac {1}{4}},\,{\tfrac {3}{10}},\,{\tfrac {7}{20}},\,{\tfrac {2}{5}},\,{\tfrac {9}{20}},\,{\tfrac {1}{2}},\,{\tfrac {11}{20}},\,{\tfrac {3}{5}},\,{\tfrac {13}{20}},\,{\tfrac {7}{10}},\,{\tfrac {3}{4}},\,{\tfrac {4}{5}},\,{\tfrac {17}{20}},\,{\tfrac {9}{10}},\,{\tfrac {19}{20}},\,{\tfrac {1}{1}}} $$

These twenty fractions are all the positive ⁠ k / d ⁠ ≤ 1 whose denominators are the divisors d = 1, 2, 4, 5, 10, 20. The fractions with 20 as denominator are those with numerators relatively prime to 20, namely ⁠ 1 / 20 ⁠, ⁠ 3 / 20 ⁠, ⁠ 7 / 20 ⁠, ⁠ 9 / 20 ⁠, ⁠ 11 / 20 ⁠, ⁠ 13 / 20 ⁠, ⁠ 17 / 20 ⁠, ⁠ 19 / 20 ⁠; by definition this is φ (20) fractions. Similarly, there are φ (10) fractions with denominator 10, and φ (5) fractions with denominator 5, etc. Thus the set of twenty fractions is split into subsets of size φ ( d ) for each d dividing 20. A similar argument applies for any n.

Möbius inversion applied to the divisor sum formula gives

$$ {\displaystyle \varphi (n)=\sum _{d\mid n}\mu \left(d\right)\cdot {\frac {n}{d}}=n\sum _{d\mid n}{\frac {\mu (d)}{d}},} $$

where μ is the Möbius function, the multiplicative function defined by ${\displaystyle \mu (p)=-1}$ and ${\displaystyle \mu (p^{k})=0}$ for each prime p and k ≥ 2. This formula may also be derived from the product formula by multiplying out ${\textstyle \prod _{p\mid n}(1-{\frac {1}{p}})}$ to get ${\textstyle \sum _{d\mid n}{\frac {\mu (d)}{d}}.}$

An example: ${\displaystyle {\begin{aligned}\varphi (20)&=\mu (1)\cdot 20+\mu (2)\cdot 10+\mu (4)\cdot 5+\mu (5)\cdot 4+\mu (10)\cdot 2+\mu (20)\cdot 1\\[.5em]&=1\cdot 20-1\cdot 10+0\cdot 5-1\cdot 4+1\cdot 2+0\cdot 1=8.\end{aligned}}}$

## Some values

The first 100 values (sequence A000010 in the OEIS ) are shown in the table and graph below:

φ ( n ) for 1 ≤ n ≤ 100 + 1 2 3 4 5 6 7 8 9 10 0 1 1 2 2 4 2 6 4 6 4 10 10 4 12 6 8 8 16 6 18 8 20 12 10 22 8 20 12 18 12 28 8 30 30 16 20 16 24 12 36 18 24 16 40 40 12 42 20 24 22 46 16 42 20 50 32 24 52 18 40 24 36 28 58 16 60 60 30 36 32 48 20 66 32 44 24 70 70 24 72 36 40 36 60 24 78 32 80 54 40 82 24 64 42 56 40 88 24 90 72 44 60 46 72 32 96 42 60 40

In the graph at right the top line y = n − 1 is an upper bound valid for all n other than one, and attained if and only if n is a prime number. A simple lower bound is ${\displaystyle \varphi (n)\geq {\sqrt {n/2}}}$, which is rather loose: in fact, the lower limit of the graph is proportional to ⁠ n / log log n ⁠.

## Euler's theorem

This states that if a and n are relatively prime then

$$ {\displaystyle a^{\varphi (n)}\equiv 1\mod n.} $$

The special case where n is prime is known as Fermat's little theorem.

This follows from Lagrange's theorem and the fact that φ ( n ) is the order of the multiplicative group of integers modulo n.

The RSA cryptosystem is based on this theorem: it implies that the inverse of the function a ↦ a mod n, where e is the (public) encryption exponent, is the function b ↦ b mod n, where d, the (private) decryption exponent, is the multiplicative inverse of e modulo φ ( n ). The difficulty of computing φ ( n ) without knowing the factorization of n is thus the difficulty of computing d: this is known as the RSA problem which can be solved by factoring n. The owner of the private key knows the factorization, since an RSA private key is constructed by choosing n as the product of two (randomly chosen) large primes p and q. Only n is publicly disclosed, and given the difficulty to factor large numbers we have the guarantee that no one else knows the factorization.

## Other formulae

- ${\displaystyle a\mid b\implies \varphi (a)\mid \varphi (b)}$
- ${\displaystyle m\mid \varphi (a^{m}-1)}$
- ${\displaystyle \varphi (mn)=\varphi (m)\varphi (n)\cdot {\frac {d}{\varphi (d)}}\quad {\text{where }}d=\operatorname {gcd} (m,n)}$ In particular:
- ${\displaystyle \varphi (2m)={\begin{cases}2\varphi (m)&{\text{ if }}m{\text{ is even}}\\\varphi (m)&{\text{ if }}m{\text{ is odd}}\end{cases}}}$
- ${\displaystyle \varphi \left(n^{m}\right)=n^{m-1}\varphi (n)}$
- ${\displaystyle \varphi (\operatorname {lcm} (m,n))\cdot \varphi (\operatorname {gcd} (m,n))=\varphi (m)\cdot \varphi (n)}$

${\displaystyle a\mid b\implies \varphi (a)\mid \varphi (b)}$

${\displaystyle m\mid \varphi (a^{m}-1)}$

${\displaystyle \varphi (mn)=\varphi (m)\varphi (n)\cdot {\frac {d}{\varphi (d)}}\quad {\text{where }}d=\operatorname {gcd} (m,n)}$ In particular:

- In particular:

In particular:

${\displaystyle \varphi (2m)={\begin{cases}2\varphi (m)&{\text{ if }}m{\text{ is even}}\\\varphi (m)&{\text{ if }}m{\text{ is odd}}\end{cases}}}$

${\displaystyle \varphi \left(n^{m}\right)=n^{m-1}\varphi (n)}$

${\displaystyle \varphi (\operatorname {lcm} (m,n))\cdot \varphi (\operatorname {gcd} (m,n))=\varphi (m)\cdot \varphi (n)}$

Compare this to the formula ${\textstyle \operatorname {lcm} (m,n)\cdot \operatorname {gcd} (m,n)=m\cdot n}$ (see least common multiple ).

Compare this to the formula ${\textstyle \operatorname {lcm} (m,n)\cdot \operatorname {gcd} (m,n)=m\cdot n}$ (see least common multiple ).

- φ ( n ) is even for n ≥ 3. Moreover, if n has r distinct odd prime factors, 2 | φ ( n )
- For any a > 1 and n > 6 such that 4 ∤ n there exists an l ≥ 2 n such that l | φ ( a − 1).
- ${\displaystyle {\frac {\varphi (n)}{n}}={\frac {\varphi (\operatorname {rad} (n))}{\operatorname {rad} (n)}}}$

φ ( n ) is even for n ≥ 3. Moreover, if n has r distinct odd prime factors, 2 | φ ( n )

For any a > 1 and n > 6 such that 4 ∤ n there exists an l ≥ 2 n such that l | φ ( a − 1).

${\displaystyle {\frac {\varphi (n)}{n}}={\frac {\varphi (\operatorname {rad} (n))}{\operatorname {rad} (n)}}}$

where rad( n ) is the radical of n (the product of all distinct primes dividing n ).

where rad( n ) is the radical of n (the product of all distinct primes dividing n ).

- ${\displaystyle \sum _{d\mid n}{\frac {\mu ^{2}(d)}{\varphi (d)}}={\frac {n}{\varphi (n)}}}$
- ${\displaystyle \sum _{1\leq k\leq n-1 \atop gcd(k,n)=1}\!\!k={\tfrac {1}{2}}n\varphi (n)\quad {\text{for }}n>1}$
- ${\displaystyle \sum _{k=1}^{n}\varphi (k)={\tfrac {1}{2}}\left(1+\sum _{k=1}^{n}\mu (k)\left\lfloor {\frac {n}{k}}\right\rfloor ^{2}\right)={\frac {3}{\pi ^{2}}}n^{2}+O\left(n(\log n)^{\frac {2}{3}}(\log \log n)^{\frac {4}{3}}\right)}$ ( cited in )
- ${\displaystyle \sum _{k=1}^{n}\varphi (k)={\frac {3}{\pi ^{2}}}n^{2}+O\left(n(\log n)^{\frac {2}{3}}(\log \log n)^{\frac {1}{3}}\right)}$ [Liu (2016)]
- ${\displaystyle \sum _{k=1}^{n}{\frac {\varphi (k)}{k}}=\sum _{k=1}^{n}{\frac {\mu (k)}{k}}\left\lfloor {\frac {n}{k}}\right\rfloor ={\frac {6}{\pi ^{2}}}n+O\left((\log n)^{\frac {2}{3}}(\log \log n)^{\frac {4}{3}}\right)}$
- ${\displaystyle \sum _{k=1}^{n}{\frac {k}{\varphi (k)}}={\frac {315\,\zeta (3)}{2\pi ^{4}}}n-{\frac {\log n}{2}}+O\left((\log n)^{\frac {2}{3}}\right)}$
- ${\displaystyle \sum _{k=1}^{n}{\frac {1}{\varphi (k)}}={\frac {315\,\zeta (3)}{2\pi ^{4}}}\left(\log n+\gamma -\sum _{p{\text{ prime}}}{\frac {\log p}{p^{2}-p+1}}\right)+O\left({\frac {(\log n)^{\frac {2}{3}}}{n}}\right)}$ (where γ is the Euler–Mascheroni constant ).

${\displaystyle \sum _{d\mid n}{\frac {\mu ^{2}(d)}{\varphi (d)}}={\frac {n}{\varphi (n)}}}$

${\displaystyle \sum _{1\leq k\leq n-1 \atop gcd(k,n)=1}\!\!k={\tfrac {1}{2}}n\varphi (n)\quad {\text{for }}n>1}$

${\displaystyle \sum _{k=1}^{n}\varphi (k)={\tfrac {1}{2}}\left(1+\sum _{k=1}^{n}\mu (k)\left\lfloor {\frac {n}{k}}\right\rfloor ^{2}\right)={\frac {3}{\pi ^{2}}}n^{2}+O\left(n(\log n)^{\frac {2}{3}}(\log \log n)^{\frac {4}{3}}\right)}$ ( cited in )

${\displaystyle \sum _{k=1}^{n}\varphi (k)={\frac {3}{\pi ^{2}}}n^{2}+O\left(n(\log n)^{\frac {2}{3}}(\log \log n)^{\frac {1}{3}}\right)}$ [Liu (2016)]

${\displaystyle \sum _{k=1}^{n}{\frac {\varphi (k)}{k}}=\sum _{k=1}^{n}{\frac {\mu (k)}{k}}\left\lfloor {\frac {n}{k}}\right\rfloor ={\frac {6}{\pi ^{2}}}n+O\left((\log n)^{\frac {2}{3}}(\log \log n)^{\frac {4}{3}}\right)}$

${\displaystyle \sum _{k=1}^{n}{\frac {k}{\varphi (k)}}={\frac {315\,\zeta (3)}{2\pi ^{4}}}n-{\frac {\log n}{2}}+O\left((\log n)^{\frac {2}{3}}\right)}$

${\displaystyle \sum _{k=1}^{n}{\frac {1}{\varphi (k)}}={\frac {315\,\zeta (3)}{2\pi ^{4}}}\left(\log n+\gamma -\sum _{p{\text{ prime}}}{\frac {\log p}{p^{2}-p+1}}\right)+O\left({\frac {(\log n)^{\frac {2}{3}}}{n}}\right)}$ (where γ is the Euler–Mascheroni constant ).

## Menon's identity

In 1965 P. Kesava Menon proved

$$ {\displaystyle \sum _{\stackrel {1\leq k\leq n}{\gcd(k,n)=1}}\!\!\!\!\gcd(k-1,n)=\varphi (n)d(n),} $$

where d ( n ) = σ 0 ( n ) is the number of divisors of n.

## Divisibility by any fixed positive integer

The following property, which is unpublished as a specific result but has long been known, has important consequences. For instance it rules out uniform distribution of the values of ${\displaystyle \varphi (n)}$ in the arithmetic progressions modulo ${\displaystyle q}$ for any integer ${\displaystyle q>1}$.

- For every fixed positive integer ${\displaystyle q}$, the relation ${\displaystyle q|\varphi (n)}$ holds for almost all ${\displaystyle n}$, meaning for all but ${\displaystyle o(x)}$ values of ${\displaystyle n\leq x}$ as ${\displaystyle x\rightarrow \infty }$.

For every fixed positive integer ${\displaystyle q}$, the relation ${\displaystyle q|\varphi (n)}$ holds for almost all ${\displaystyle n}$, meaning for all but ${\displaystyle o(x)}$ values of ${\displaystyle n\leq x}$ as ${\displaystyle x\rightarrow \infty }$.

This is an elementary consequence of the fact that the sum of the reciprocals of the primes congruent to 1 modulo ${\displaystyle q}$ diverges, which itself is a corollary of the proof of Dirichlet's theorem on arithmetic progressions.

## Generating functions

The Dirichlet series for φ ( n ) may be written in terms of the Riemann zeta function as:

$$ {\displaystyle \sum _{n=1}^{\infty }{\frac {\varphi (n)}{n^{s}}}={\frac {\zeta (s-1)}{\zeta (s)}}} $$

where the left-hand side converges for ${\displaystyle \Re (s)>2}$.

The Lambert series generating function is

$$ {\displaystyle \sum _{n=1}^{\infty }{\frac {\varphi (n)q^{n}}{1-q^{n}}}={\frac {q}{(1-q)^{2}}}} $$

which converges for | q | < 1.

Both of these are proved by elementary series manipulations and the formulae for φ ( n ).

## Growth rate

In the words of Hardy & Wright, the order of φ ( n ) is "always 'nearly n '."

First

$$ {\displaystyle \lim \sup {\frac {\varphi (n)}{n}}=1,} $$

but as n goes to infinity, for all δ > 0

$$ {\displaystyle {\frac {\varphi (n)}{n^{1-\delta }}}\rightarrow \infty.} $$

These two formulae can be proved by using little more than the formulae for φ ( n ) and the divisor sum function σ ( n ).

In fact, during the proof of the second formula, the inequality

$$ {\displaystyle {\frac {6}{\pi ^{2}}}<{\frac {\varphi (n)\sigma (n)}{n^{2}}}<1,} $$

true for n > 1, is proved.

We also have

$$ {\displaystyle \lim \inf {\frac {\varphi (n)}{n}}\log \log n=e^{-\gamma }.} $$

Here γ is Euler's constant, γ = 0.577215665..., so e = 1.7810724... and e = 0.56145948....

Proving this does not quite require the prime number theorem. Since log log n goes to infinity, this formula shows that

$$ {\displaystyle \lim \inf {\frac {\varphi (n)}{n}}=0.} $$

In fact, more is true.

$$ {\displaystyle \varphi (n)>{\frac {n}{e^{\gamma }\;\log \log n+{\frac {3}{\log \log n}}}}\quad {\text{for }}n>2} $$

and

$$ {\displaystyle \varphi (n)<{\frac {n}{e^{\gamma }\log \log n}}\quad {\text{for infinitely many }}n.} $$

The second inequality was shown by Jean-Louis Nicolas. Ribenboim says "The method of proof is interesting, in that the inequality is shown first under the assumption that the Riemann hypothesis is true, secondly under the contrary assumption."

For the average order, we have

$$ {\displaystyle \varphi (1)+\varphi (2)+\cdots +\varphi (n)={\frac {3n^{2}}{\pi ^{2}}}+O\left(n(\log n)^{\frac {2}{3}}(\log \log n)^{\frac {4}{3}}\right)\quad {\text{as }}n\rightarrow \infty,} $$

due to Arnold Walfisz, its proof exploiting estimates on exponential sums due to I. M. Vinogradov and N. M. Korobov. By a combination of van der Corput's and Vinogradov's methods, H.-Q. Liu (On Euler's function.Proc. Roy. Soc. Edinburgh Sect. A 146 (2016), no. 4, 769–775) improved the error term to

$$ {\displaystyle O\left(n(\log n)^{\frac {2}{3}}(\log \log n)^{\frac {1}{3}}\right)} $$

(this is currently the best known estimate of this type). The "Big O " stands for a quantity that is bounded by a constant times the function of n inside the parentheses (which is small compared to n ).

This result can be used to prove that the probability of two randomly chosen numbers being relatively prime is ⁠ 6 / π ⁠.

## Ratio of consecutive values

In 1950 Somayajulu proved

$$ {\displaystyle {\begin{aligned}\lim \inf {\frac {\varphi (n+1)}{\varphi (n)}}&=0\quad {\text{and}}\\[5px]\lim \sup {\frac {\varphi (n+1)}{\varphi (n)}}&=\infty.\end{aligned}}} $$

In 1954 Schinzel and Sierpiński strengthened this, proving that the set

$$ {\displaystyle \left\{{\frac {\varphi (n+1)}{\varphi (n)}},\;\;n=1,2,\ldots \right\}} $$

is dense in the positive real numbers. They also proved that the set

$$ {\displaystyle \left\{{\frac {\varphi (n)}{n}},\;\;n=1,2,\ldots \right\}} $$

is dense in the interval (0,1).

## Totient number

A totient number is a value of Euler's totient function: that is, an m for which there is at least one n for which φ ( n ) = m. The valency or multiplicity of a totient number m is the number of solutions to this equation. A nontotient is a natural number which is not a totient number. Every odd integer exceeding 1 is trivially a nontotient. There are also infinitely many even nontotients, and indeed every positive integer has a multiple which is an even nontotient.

The first few totient numbers are ${\displaystyle 1,2,4,6,8,10,12,16,18,20}$, see sequence A002202.

The number of totient numbers up to a given limit x is

$$ {\displaystyle {\frac {x}{\log x}}e^{{\big (}C+o(1){\big )}(\log \log \log x)^{2}}} $$

for a constant C = 0.8178146....

If counted accordingly to multiplicity, the number of totient numbers up to a given limit x is

$$ {\displaystyle {\Big \vert }\{n:\varphi (n)\leq x\}{\Big \vert }={\frac {\zeta (2)\zeta (3)}{\zeta (6)}}\cdot x+R(x)} $$

where the error term R is of order at most ⁠ x / (log x ) ⁠ for any positive k.

It is known that the multiplicity of m exceeds m infinitely often for any δ < 0.55655.

## Ford's theorem

Ford (1999) proved that for every integer k ≥ 2 there is a totient number m of multiplicity k: that is, for which the equation φ ( n ) = m has exactly k solutions; this result had previously been conjectured by Wacław Sierpiński, and it had been obtained as a consequence of Schinzel's hypothesis H. Indeed, each multiplicity that occurs, does so infinitely often.

However, no number m is known with multiplicity k = 1. Carmichael's totient function conjecture is the statement that there is no such m.

## Perfect totient numbers

A perfect totient number is an integer that is equal to the sum of its iterated totients. That is, we apply the totient function to a number n, apply it again to the resulting totient, and so on, until the number 1 is reached, and add together the resulting sequence of numbers; if the sum equals n, then n is a perfect totient number.

## Applications

## Cyclotomy

In the last section of the Disquisitiones Gauss proves that a regular n -gon can be constructed with straightedge and compass if φ ( n ) is a power of 2. If n is a power of an odd prime number the formula for the totient says its totient can be a power of two only if n is a first power and n − 1 is a power of 2. The primes that are one more than a power of 2 are called Fermat primes, and only five are known: 3, 5, 17, 257, and 65537. Fermat and Gauss knew of these. Nobody has been able to prove whether there are any more.

Thus, a regular n -gon has a straightedge-and-compass construction if n is a product of distinct Fermat primes and any power of 2. The first few such n are

2, 3, 4, 5, 6, 8, 10, 12, 15, 16, 17, 20, 24, 30, 32, 34, 40,... (sequence A003401 in the OEIS ).

## Prime number theorem for arithmetic progressions

## The RSA cryptosystem

Setting up an RSA system involves choosing large prime numbers p and q, computing n = pq and k = φ ( n ), and finding two numbers e and d such that ed ≡ 1 (mod k ). The numbers n and e (the "encryption key") are released to the public, and d (the "decryption key") is kept private.

A message, represented by an integer m, where 0 < m < n, is encrypted by computing S = m (mod n ).

It is decrypted by computing t = S (mod n ). Euler's Theorem can be used to show that if 0 < t < n, then t = m.

The security of an RSA system would be compromised if the number n could be efficiently factored or if φ ( n ) could be efficiently computed without factoring n.

## Unsolved problems

## Lehmer's conjecture

If p is prime, then φ ( p ) = p − 1. In 1932 D. H. Lehmer asked if there are any composite numbers n such that φ ( n ) divides n − 1. None are known.

In 1933 he proved that if any such n exists, it must be odd, square-free, and divisible by at least seven primes (i.e. ω ( n ) ≥ 7 ). In 1980 Cohen and Hagis proved that n > 10 and that ω ( n ) ≥ 14. Further, Hagis showed that if 3 divides n then n > 10 and ω ( n ) ≥ 298848.

## Carmichael's conjecture

This states that there is no number ${\displaystyle n}$ with the property that for all other numbers ${\displaystyle m}$, ${\displaystyle m\neq n}$, ${\displaystyle \varphi (m)\neq \varphi (n)}$. See Ford's theorem above.

If there is a single counterexample to this conjecture, there must be infinitely many counterexamples, and the smallest one has at least ten billion digits in base 10.

## Riemann hypothesis

The Riemann hypothesis is true if and only if the inequality

$$ {\displaystyle {\frac {n}{\varphi (n)}}<e^{\gamma }\log \log n+{\frac {e^{\gamma }(4+\gamma -\log 4\pi )}{\sqrt {\log n}}}} $$

is true for all ${\displaystyle n\geq p_{120569}\#}$ where ${\displaystyle \gamma }$ is Euler's constant and ${\displaystyle p_{120569}\#}$ is the product of the first 120569 primes.


------------------------------------------------------------------

In [16]:
from display_helpers import show_page_by_title

# Pretty print page given a title

show_page_by_title(stored_normal_wiki_data, "Bose–Einstein statistics")


# Bose–Einstein statistics

## Introduction

- Thermodynamics
- Kinetic theory

Thermodynamics

Kinetic theory

In quantum statistics, Bose–Einstein statistics ( B–E statistics ) describes one of two possible ways in which a collection of non-interacting identical particles may occupy a set of available discrete energy states at thermodynamic equilibrium. The aggregation of particles in the same state, which is a characteristic of particles obeying Bose–Einstein statistics, accounts for the cohesive streaming of laser light and the frictionless creeping of superfluid helium. The theory of this behaviour was developed (1924–25) by Satyendra Nath Bose, who recognized that a collection of identical and indistinguishable particles could be distributed in this way. The idea was later adopted and extended by Albert Einstein in collaboration with Bose.

Bose–Einstein statistics apply only to particles that do not follow the Pauli exclusion principle restrictions. Particles that follow Bose-Einstein statistics are called bosons, which have integer values of spin. In contrast, particles that follow Fermi-Dirac statistics are called fermions and have half-integer spins.

## Bose–Einstein distribution

At low temperatures, bosons behave differently from fermions (which obey the Fermi–Dirac statistics ) in a way that an unlimited number of them can "condense" into the same energy state. This apparently unusual property also gives rise to the special state of matter – the Bose–Einstein condensate. Fermi–Dirac and Bose–Einstein statistics apply when quantum effects are important and the particles are " indistinguishable ". Quantum effects appear if the concentration of particles satisfies ${\displaystyle {\frac {N}{V}}\geq n_{\text{q}},}$ where N is the number of particles, V is the volume, and n q is the quantum concentration, for which the interparticle distance is equal to the thermal de Broglie wavelength, so that the wavefunctions of the particles are barely overlapping.

Fermi–Dirac statistics applies to fermions (particles that obey the Pauli exclusion principle ), and Bose–Einstein statistics applies to bosons. As the quantum concentration depends on temperature, most systems at high temperatures obey the classical (Maxwell–Boltzmann) limit, unless they also have a very high density, as for a white dwarf. Both Fermi–Dirac and Bose–Einstein become Maxwell–Boltzmann statistics at high temperature or at low concentration.

Bose–Einstein statistics was introduced for photons in 1924 by Bose and generalized to atoms by Einstein in 1924–25.

The expected number of particles in an energy state i for Bose–Einstein statistics is:

$$ {\displaystyle {\bar {n}}_{i}={\frac {g_{i}}{e^{(\varepsilon _{i}-\mu )/k_{\text{B}}T}-1}}} $$

with ε i > μ and where n i is the occupation number (the number of particles) in state i, ${\displaystyle g_{i}}$ is the degeneracy of energy level i, ε i is the energy of the i th state, μ is the chemical potential (zero for a photon gas ), k B is the Boltzmann constant, and T is the absolute temperature.

The variance of this distribution ${\displaystyle V(n)}$ is calculated directly from the expression above for the average number. ${\displaystyle V(n)=kT{\frac {\partial }{\partial \mu }}{\bar {n}}_{i}=\langle n\rangle (1+\langle n\rangle )={\bar {n}}+{\bar {n}}^{2}}$

For comparison, the average number of fermions with energy ${\displaystyle \varepsilon _{i}}$ given by Fermi–Dirac particle-energy distribution has a similar form: ${\displaystyle {\bar {n}}_{i}(\varepsilon _{i})={\frac {g_{i}}{e^{(\varepsilon _{i}-\mu )/k_{\text{B}}T}+1}}.}$

As mentioned above, both the Bose–Einstein distribution and the Fermi–Dirac distribution approaches the Maxwell–Boltzmann distribution in the limit of high temperature and low particle density, without the need for any ad hoc assumptions:

- In the limit of low particle density, ${\displaystyle {\bar {n}}_{i}={\frac {g_{i}}{e^{(\varepsilon _{i}-\mu )/k_{\text{B}}T}\pm 1}}\ll 1}$, therefore ${\displaystyle e^{(\varepsilon _{i}-\mu )/k_{\text{B}}T}\pm 1\gg 1}$ or equivalently ${\displaystyle e^{(\varepsilon _{i}-\mu )/k_{\text{B}}T}\gg 1}$. In that case, ${\displaystyle {\bar {n}}_{i}\approx {\frac {g_{i}}{e^{(\varepsilon _{i}-\mu )/k_{\text{B}}T}}}={\frac {1}{Z}}e^{-(\varepsilon _{i}-\mu )/k_{\text{B}}T}}$, which is the result from Maxwell–Boltzmann statistics.
- In the limit of high temperature, the particles are distributed over a large range of energy values, therefore the occupancy on each state (especially the high energy ones with ${\displaystyle \varepsilon _{i}-\mu \gg k_{\text{B}}T}$ ) is again very small, ${\displaystyle {\bar {n}}_{i}={\frac {g_{i}}{e^{(\varepsilon _{i}-\mu )/k_{\text{B}}T}\pm 1}}\ll 1}$. This again reduces to Maxwell–Boltzmann statistics.

In the limit of low particle density, ${\displaystyle {\bar {n}}_{i}={\frac {g_{i}}{e^{(\varepsilon _{i}-\mu )/k_{\text{B}}T}\pm 1}}\ll 1}$, therefore ${\displaystyle e^{(\varepsilon _{i}-\mu )/k_{\text{B}}T}\pm 1\gg 1}$ or equivalently ${\displaystyle e^{(\varepsilon _{i}-\mu )/k_{\text{B}}T}\gg 1}$. In that case, ${\displaystyle {\bar {n}}_{i}\approx {\frac {g_{i}}{e^{(\varepsilon _{i}-\mu )/k_{\text{B}}T}}}={\frac {1}{Z}}e^{-(\varepsilon _{i}-\mu )/k_{\text{B}}T}}$, which is the result from Maxwell–Boltzmann statistics.

In the limit of high temperature, the particles are distributed over a large range of energy values, therefore the occupancy on each state (especially the high energy ones with ${\displaystyle \varepsilon _{i}-\mu \gg k_{\text{B}}T}$ ) is again very small, ${\displaystyle {\bar {n}}_{i}={\frac {g_{i}}{e^{(\varepsilon _{i}-\mu )/k_{\text{B}}T}\pm 1}}\ll 1}$. This again reduces to Maxwell–Boltzmann statistics.

In addition to reducing to the Maxwell–Boltzmann distribution in the limit of high ${\displaystyle T}$ and low density, Bose–Einstein statistics also reduces to Rayleigh–Jeans law distribution for low energy states with ${\displaystyle \varepsilon _{i}-\mu \ll k_{\text{B}}T}$, namely ${\displaystyle {\begin{aligned}{\bar {n}}_{i}&={\frac {g_{i}}{e^{(\varepsilon _{i}-\mu )/k_{\text{B}}T}-1}}\\&\approx {\frac {g_{i}}{(\varepsilon _{i}-\mu )/k_{\text{B}}T}}={\frac {g_{i}k_{\text{B}}T}{\varepsilon _{i}-\mu }}.\end{aligned}}}$

## History

In 1900, Max Planck derived the Planck law to explain blackbody radiation. For this purpose, he introduced the concept of quanta of energy.

Władysław Natanson in 1911 concluded that Planck's law requires indistinguishability of "units of energy", although he did not frame this in terms of Einstein's light quanta.

While presenting a lecture at the University of Dhaka (in what was then British India and is now Bangladesh ) on the theory of radiation and the ultraviolet catastrophe, Satyendra Nath Bose intended to show his students that the contemporary theory was inadequate, because it predicted results not in accordance with experimental results. During this lecture, Bose committed an error in applying the theory, which unexpectedly gave a prediction that agreed with the experiment. The error was a simple mistake – similar to arguing that flipping two fair coins will produce two heads one-third of the time – that would appear obviously wrong to anyone with a basic understanding of statistics (remarkably, this error resembled the famous blunder by Jean le Rond d'Alembert known from his Croix ou Pile article ). However, the results it predicted agreed with experiment, and Bose realized it might not be a mistake after all. For the first time, he took the position that the Maxwell–Boltzmann distribution would not be true for all microscopic particles at all scales. Thus, he studied the probability of finding particles in various states in phase space, where each state is a little patch having phase volume of h, and the position and momentum of the particles are not kept particularly separate but are considered as one variable.

Bose adapted this lecture into a short article called "Planck's law and the hypothesis of light quanta" and submitted it to the Philosophical Magazine. However, the referee's report was negative, and the paper was rejected. Undaunted, he sent the manuscript to Albert Einstein requesting publication in the Zeitschrift für Physik. Einstein immediately agreed, personally translated the article from English into German (Bose had earlier translated Einstein's article on the general theory of relativity from German to English), and saw to it that it was published. Bose's theory achieved respect when Einstein sent his own paper in support of Bose's to Zeitschrift für Physik, asking that they be published together. The paper came out in 1924.

The reason Bose produced accurate results was that since photons are indistinguishable from each other, one cannot treat any two photons having equal quantum numbers (e.g., polarization and momentum vector) as being two distinct identifiable photons. Bose originally had a factor of 2 for the possible spin states, but Einstein changed it to polarization. By analogy, if in an alternate universe coins were to behave like photons and other bosons, the probability of producing two heads would indeed be one-third, and so is the probability of getting a head and a tail which equals one-half for the conventional (classical, distinguishable) coins. Bose's "error" leads to what is now called Bose–Einstein statistics.

Bose and Einstein extended the idea to atoms and this led to the prediction of the existence of phenomena which became known as Bose–Einstein condensate, a dense collection of bosons (which are particles with integer spin, named after Bose), which was demonstrated to exist by experiment in 1995.

## Derivation

## Derivation from the microcanonical ensemble

In the microcanonical ensemble, one considers a system with fixed energy, volume, and number of particles. We take a system composed of ${\textstyle N=\sum _{i}n_{i}}$ identical bosons, ${\displaystyle n_{i}}$ of which have energy ${\displaystyle \varepsilon _{i}}$ and are distributed over ${\displaystyle g_{i}}$ levels or states with the same energy ${\displaystyle \varepsilon _{i}}$, i.e. ${\displaystyle g_{i}}$ is the degeneracy associated with energy ${\displaystyle \varepsilon _{i}}$. The total energy of the system is ${\textstyle E=\sum _{i}n_{i}\varepsilon _{i}}$. Calculation of the number of arrangements of ${\displaystyle n_{i}}$ particles distributed among ${\displaystyle g_{i}}$ states is a problem of combinatorics. Since particles are indistinguishable in the quantum mechanical context here, the number of ways for arranging ${\displaystyle n_{i}}$ particles in ${\displaystyle g_{i}}$ boxes (for the ${\displaystyle i}$ th energy level), where each box is capable of containing an infinite number of bosons (because for bosons the Pauli exclusion principle does not apply), would be (see image):

${\displaystyle w_{i,{\text{BE}}}={\frac {(n_{i}+g_{i}-1)!}{n_{i}!(g_{i}-1)!}}=C_{n_{i}}^{n_{i}+g_{i}-1},}$ where ${\displaystyle C_{k}^{m}}$ is the k -combination of a set with m elements (Note also that ${\displaystyle w_{i,{\text{BE}}}}$ represents the absolute non-normalized probability of an energy state with ${\displaystyle n_{i}}$ bosons and a degeneracy of ${\displaystyle g_{i}}$, it is not the same as the ${\displaystyle w_{i}}$ associated with the Gibbs formulation of entropy). The total number of arrangements in an ensemble of bosons is simply the product of the binomial coefficients ${\displaystyle C_{n_{i}}^{n_{i}+g_{i}-1}}$ above over all the energy levels, i.e. ${\displaystyle W_{\text{BE}}=\prod _{i}w_{i,{\text{BE}}}=\prod _{i}{\frac {(n_{i}+g_{i}-1)!}{(g_{i}-1)!n_{i}!}},}$

which for very large ${\displaystyle n_{i}}$ and ${\displaystyle g_{i}}$ can be simplified using Stirling's approximation to

$$ {\displaystyle W_{\text{BE}}=\prod _{i}{\frac {({\frac {n_{i}+g_{i}-1}{e}})^{n_{i}+g_{i}-1}}{({\frac {g_{i}-1}{e}})^{g_{i}-1}({\frac {n_{i}}{e}})^{n_{i}}}}.} $$

The entropy of the system can then be expressed as

$$ {\displaystyle S_{\text{BE}}=k_{B}{\text{ln}}W_{\text{BE}}=k_{B}\sum _{i}[(n_{i}+g_{i}-1)({\text{ln}}(n_{i}+g_{i}-1)-1)-(g_{i}-1)({\text{ln}}(g_{i}-1)-1)-n_{i}({\text{ln}}n_{i}-1)].} $$

The three constraints we can impose on the system can be expressed as

$$ {\displaystyle \sum _{i}\delta n_{i}=0} $$

(conservation of N),

$$ {\displaystyle \sum _{i}\epsilon _{i}\delta n_{i}=0} $$

(conservation of E), and

$$ {\displaystyle \delta S_{\text{BE}}=0} $$

( second law of thermodynamics for a system at equilibrium).

This final constraint can be expanded to be in terms of ${\displaystyle n_{i}}$:

$$ {\displaystyle \delta S_{\text{BE}}={\frac {\partial }{\partial n_{i}}}S_{\text{BE}}\delta n_{i}=k_{B}\sum _{i}[{\text{ln}}(n_{i}+g_{i}-1)-{\text{ln}}n_{i}]\delta n_{i}=0.} $$

Now we can write

$$ {\displaystyle \sum _{i}[{\text{ln}}(n_{i}+g_{i}-1)-{\text{ln}}n_{i}]\delta n_{i}+C\sum _{i}\delta n_{i}-\beta \sum _{i}\epsilon _{i}\delta n_{i}=0,} $$

for which to be true, it must be the case that for any i

$$ {\displaystyle {\text{ln}}(n_{i}+g_{i}-1)-{\text{ln}}n_{i}+C-\beta \epsilon _{i}=0.} $$

By solving for ${\displaystyle n_{i}}$ and simplifying we obtain

$$ {\displaystyle n_{i}={\frac {g_{i}-1}{\alpha e^{\beta \epsilon _{i}}-1}},} $$

which for sufficiently large ${\displaystyle g_{i}}$ reduces to

$$ {\displaystyle n_{i}={\frac {g_{i}}{\alpha e^{\beta \epsilon _{i}}-1}},} $$

which is the form of the Bose-Einstein distribution. Note that this form holds even for a system of interacting bosons.

## Derivation from the grand canonical ensemble

The Bose–Einstein distribution, which applies only to a quantum system of non-interacting bosons, is naturally derived from the grand canonical ensemble without any approximations. In this ensemble, the system is able to exchange energy and exchange particles with a reservoir (temperature T and chemical potential μ fixed by the reservoir).

Due to the non-interacting quality, each available single-particle level (with energy level ϵ ) forms a separate thermodynamic system in contact with the reservoir. That is, the number of particles within the overall system that occupy a given single particle state form a sub-ensemble that is also grand canonical ensemble; hence, it may be analysed through the construction of a grand partition function.

Every single-particle state is of a fixed energy, ${\displaystyle \varepsilon }$. As the sub-ensemble associated with a single-particle state varies by the number of particles only, it is clear that the total energy of the sub-ensemble is also directly proportional to the number of particles in the single-particle state; where ${\displaystyle N}$ is the number of particles, the total energy of the sub-ensemble will then be ${\displaystyle N\varepsilon }$. Beginning with the standard expression for a grand partition function and replacing ${\displaystyle E}$ with ${\displaystyle N\varepsilon }$, the grand partition function takes the form ${\displaystyle {\mathcal {Z}}=\sum _{N}\exp((N\mu -N\varepsilon )/k_{\text{B}}T)=\sum _{N}\exp(N(\mu -\varepsilon )/k_{\text{B}}T)}$

This formula applies to fermionic systems as well as bosonic systems. Fermi–Dirac statistics arises when considering the effect of the Pauli exclusion principle: whilst the number of fermions occupying the same single-particle state can only be either 1 or 0, the number of bosons occupying a single particle state may be any integer. Thus, the grand partition function for bosons can be considered a geometric series and may be evaluated as such: ${\displaystyle {\begin{aligned}{\mathcal {Z}}&=\sum _{N=0}^{\infty }\exp(N(\mu -\varepsilon )/k_{\text{B}}T)=\sum _{N=0}^{\infty }[\exp((\mu -\varepsilon )/k_{\text{B}}T)]^{N}\\&={\frac {1}{1-\exp((\mu -\varepsilon )/k_{\text{B}}T)}}.\end{aligned}}}$

Note that the geometric series is convergent only if ${\displaystyle e^{(\mu -\varepsilon )/k_{\text{B}}T}<1}$, including the case where ${\displaystyle \varepsilon =0}$. This implies that the chemical potential for the Bose gas must be negative, i.e., ${\displaystyle \mu <0}$, whereas the Fermi gas is allowed to take both positive and negative values for the chemical potential.

The average particle number for that single-particle substate is given by ${\displaystyle \langle N\rangle =k_{\text{B}}T{\frac {1}{\mathcal {Z}}}\left({\frac {\partial {\mathcal {Z}}}{\partial \mu }}\right)_{V,T}={\frac {1}{\exp((\varepsilon -\mu )/k_{\text{B}}T)-1}}}$ This result applies for each single-particle level and thus forms the Bose–Einstein distribution for the entire state of the system.

The variance in particle number, ${\textstyle \sigma _{N}^{2}=\langle N^{2}\rangle -\langle N\rangle ^{2}}$, is: ${\displaystyle \sigma _{N}^{2}=k_{\text{B}}T\left({\frac {d\langle N\rangle }{d\mu }}\right)_{V,T}={\frac {\exp((\varepsilon -\mu )/k_{\text{B}}T)}{(\exp((\varepsilon -\mu )/k_{\text{B}}T)-1)^{2}}}=\langle N\rangle (1+\langle N\rangle ).}$

As a result, for highly occupied states the standard deviation of the particle number of an energy level is very large, slightly larger than the particle number itself: ${\displaystyle \sigma _{N}\approx \langle N\rangle }$. This large uncertainty is due to the fact that the probability distribution for the number of bosons in a given energy level is a geometric distribution; somewhat counterintuitively, the most probable value for N is always 0. (In contrast, classical particles have instead a Poisson distribution in particle number for a given state, with a much smaller uncertainty of ${\textstyle \sigma _{N,{\rm {classical}}}={\sqrt {\langle N\rangle }}}$, and with the most-probable N value being near ${\displaystyle \langle N\rangle }$.)

## Derivation in the canonical approach

It is also possible to derive approximate Bose–Einstein statistics in the canonical ensemble. These derivations are lengthy and only yield the above results in the asymptotic limit of a large number of particles. The reason is that the total number of bosons is fixed in the canonical ensemble. The Bose–Einstein distribution in this case can be derived as in most texts by maximization, but the mathematically best derivation is by the Darwin–Fowler method of mean values as emphasized by Dingle. See also Müller-Kirsten. The fluctuations of the ground state in the condensed region are however markedly different in the canonical and grand-canonical ensembles.

Suppose we have a number of energy levels, labeled by index ${\displaystyle i}$, each level having energy ${\displaystyle \varepsilon _{i}}$ and containing a total of ${\displaystyle n_{i}}$ particles. Suppose each level contains ${\displaystyle g_{i}}$ distinct sublevels, all of which have the same energy, and which are distinguishable. For example, two particles may have different momenta, in which case they are distinguishable from each other, yet they can still have the same energy. The value of ${\displaystyle g_{i}}$ associated with level ${\displaystyle i}$ is called the "degeneracy" of that energy level. Any number of bosons can occupy the same sublevel.

Let ${\displaystyle w(n,g)}$ be the number of ways of distributing ${\displaystyle n}$ particles among the ${\displaystyle g}$ sublevels of an energy level. There is only one way of distributing ${\displaystyle n}$ particles with one sublevel, therefore ${\displaystyle w(n,1)=1}$. It is easy to see that there are ${\displaystyle (n+1)}$ ways of distributing ${\displaystyle n}$ particles in two sublevels which we will write as: ${\displaystyle w(n,2)={\frac {(n+1)!}{n!1!}}.}$

With a little thought (see Notes below) it can be seen that the number of ways of distributing ${\displaystyle n}$ particles in three sublevels is ${\displaystyle w(n,3)=w(n,2)+w(n-1,2)+\cdots +w(1,2)+w(0,2)}$ so that ${\displaystyle w(n,3)=\sum _{k=0}^{n}w(n-k,2)=\sum _{k=0}^{n}{\frac {(n-k+1)!}{(n-k)!1!}}={\frac {(n+2)!}{n!2!}}}$ where we have used the following theorem involving binomial coefficients: ${\displaystyle \sum _{k=0}^{n}{\frac {(k+a)!}{k!a!}}={\frac {(n+a+1)!}{n!(a+1)!}}.}$

Continuing this process, we can see that ${\displaystyle w(n,g)}$ is just a binomial coefficient (See Notes below) ${\displaystyle w(n,g)={\frac {(n+g-1)!}{n!(g-1)!}}.}$

For example, the population numbers for two particles in three sublevels are 200, 110, 101, 020, 011, or 002 for a total of six which equals 4!/(2!2!). The number of ways that a set of occupation numbers ${\displaystyle n_{i}}$ can be realized is the product of the ways that each individual energy level can be populated: ${\displaystyle W=\prod _{i}w(n_{i},g_{i})=\prod _{i}{\frac {(n_{i}+g_{i}-1)!}{n_{i}!(g_{i}-1)!}}\approx \prod _{i}{\frac {(n_{i}+g_{i})!}{n_{i}!(g_{i})!}}}$ where the approximation assumes that ${\displaystyle n_{i}\gg 1}$.

Following the same procedure used in deriving the Maxwell–Boltzmann statistics, we wish to find the set of ${\displaystyle n_{i}}$ for which W is maximised, subject to the constraint that there be a fixed total number of particles, and a fixed total energy. The maxima of ${\displaystyle W}$ and ${\displaystyle \ln(W)}$ occur at the same value of ${\displaystyle n_{i}}$ and, since it is easier to accomplish mathematically, we will maximise the latter function instead. We constrain our solution using Lagrange multipliers forming the function: ${\displaystyle f(n_{i})=\ln(W)+\alpha (N-\sum n_{i})+\beta (E-\sum n_{i}\varepsilon _{i})}$

Using the ${\displaystyle n_{i}\gg 1}$ approximation and using Stirling's approximation for the factorials ${\displaystyle \left(x!\approx x^{x}\,e^{-x}\,{\sqrt {2\pi x}}\right)}$ gives ${\displaystyle f(n_{i})=\sum _{i}(n_{i}+g_{i})\ln(n_{i}+g_{i})-n_{i}\ln(n_{i})+\alpha \left(N-\sum n_{i}\right)+\beta \left(E-\sum n_{i}\varepsilon _{i}\right)+K,}$ where K is the sum of a number of terms which are not functions of the ${\displaystyle n_{i}}$. Taking the derivative with respect to ${\displaystyle n_{i}}$, and setting the result to zero and solving for ${\displaystyle n_{i}}$, yields the Bose–Einstein population numbers: ${\displaystyle n_{i}={\frac {g_{i}}{e^{\alpha +\beta \varepsilon _{i}}-1}}.}$

By a process similar to that outlined in the Maxwell–Boltzmann statistics article, it can be seen that: ${\displaystyle d\ln W=\alpha \,dN+\beta \,dE}$ which, using Boltzmann's famous relationship ${\displaystyle S=k_{\text{B}}\,\ln W}$ becomes a statement of the second law of thermodynamics at constant volume, and it follows that ${\displaystyle \beta ={\frac {1}{k_{\text{B}}T}}}$ and ${\displaystyle \alpha =-{\frac {\mu }{k_{\text{B}}T}}}$ where S is the entropy, ${\displaystyle \mu }$ is the chemical potential, k B is the Boltzmann constant and T is the temperature, so that finally: ${\displaystyle n_{i}={\frac {g_{i}}{e^{(\varepsilon _{i}-\mu )/k_{\text{B}}T}-1}}.}$

Note that the above formula is sometimes written: ${\displaystyle n_{i}={\frac {g_{i}}{e^{\varepsilon _{i}/k_{\text{B}}T}/z-1}},}$ where ${\displaystyle z=\exp(\mu /k_{\text{B}}T)}$ is the absolute activity, as noted by McQuarrie.

Also note that when the particle numbers are not conserved, removing the conservation of particle numbers constraint is equivalent to setting ${\displaystyle \alpha }$ and therefore the chemical potential ${\displaystyle \mu }$ to zero. This will be the case for photons and massive particles in mutual equilibrium and the resulting distribution will be the Planck distribution.

A much simpler way to think of Bose–Einstein distribution function is to consider that n particles are denoted by identical balls and g shells are marked by g-1 line partitions. It is clear that the permutations of these n balls and g − 1 partitions will give different ways of arranging bosons in different energy levels. Say, for 3 (= n ) particles and 3 (= g ) shells, therefore ( g − 1) = 2, the arrangement might be |●●|●, or ||●●●, or |●|●●, etc. Hence the number of distinct permutations of n + ( g − 1) objects which have n identical items and ( g − 1) identical items will be: ${\displaystyle {\frac {(g-1+n)!}{(g-1)!n!}}}$

OR

The purpose of these notes is to clarify some aspects of the derivation of the Bose–Einstein distribution for beginners. The enumeration of cases (or ways) in the Bose–Einstein distribution can be recast as follows. Consider a game of dice throwing in which there are ${\displaystyle n}$ dice, with each die taking values in the set ${\displaystyle \{1,\dots,g\}}$, for ${\displaystyle g\geq 1}$. The constraints of the game are that the value of a die ${\displaystyle i}$, denoted by ${\displaystyle m_{i}}$, has to be greater than or equal to the value of die ${\displaystyle (i-1)}$, denoted by ${\displaystyle m_{i-1}}$, in the previous throw, i.e., ${\displaystyle m_{i}\geq m_{i-1}}$. Thus a valid sequence of die throws can be described by an n -tuple ${\displaystyle (m_{1},m_{2},\dots,m_{n})}$, such that ${\displaystyle m_{i}\geq m_{i-1}}$. Let ${\displaystyle S(n,g)}$ denote the set of these valid n -tuples:

Then the quantity ${\displaystyle w(n,g)}$ ( defined above as the number of ways to distribute ${\displaystyle n}$ particles among the ${\displaystyle g}$ sublevels of an energy level) is the cardinality of ${\displaystyle S(n,g)}$, i.e., the number of elements (or valid n -tuples) in ${\displaystyle S(n,g)}$. Thus the problem of finding an expression for ${\displaystyle w(n,g)}$ becomes the problem of counting the elements in ${\displaystyle S(n,g)}$.

Example n = 4, g = 3: ${\displaystyle S(4,3)=\left\{\underbrace {(1111),(1112),(1113)} _{(a)},\underbrace {(1122),(1123),(1133)} _{(b)},\underbrace {(1222),(1223),(1233),(1333)} _{(c)},\underbrace {(2222),(2223),(2233),(2333),(3333)} _{(d)}\right\}}$ ${\displaystyle w(4,3)=15}$ (there are ${\displaystyle 15}$ elements in ${\displaystyle S(4,3)}$ )

Subset ${\displaystyle (a)}$ is obtained by fixing all indices ${\displaystyle m_{i}}$ to ${\displaystyle 1}$, except for the last index, ${\displaystyle m_{n}}$, which is incremented from ${\displaystyle 1}$ to ${\displaystyle g=3}$. Subset ${\displaystyle (b)}$ is obtained by fixing ${\displaystyle m_{1}=m_{2}=1}$, and incrementing ${\displaystyle m_{3}}$ from ${\displaystyle 2}$ to ${\displaystyle g=3}$. Due to the constraint ${\displaystyle m_{i}\geq m_{i-1}}$ on the indices in ${\displaystyle S(n,g)}$, the index ${\displaystyle m_{4}}$ must automatically take values in ${\displaystyle \left\{2,3\right\}}$. The construction of subsets ${\displaystyle (c)}$ and ${\displaystyle (d)}$ follows in the same manner.

Each element of ${\displaystyle S(4,3)}$ can be thought of as a multiset of cardinality ${\displaystyle n=4}$; the elements of such multiset are taken from the set ${\displaystyle \left\{1,2,3\right\}}$ of cardinality ${\displaystyle g=3}$, and the number of such multisets is the multiset coefficient ${\displaystyle \left\langle {\begin{matrix}3\\4\end{matrix}}\right\rangle ={3+4-1 \choose 3-1}={3+4-1 \choose 4}={\frac {6!}{4!2!}}=15}$

More generally, each element of ${\displaystyle S(n,g)}$ is a multiset of cardinality ${\displaystyle n}$ (number of dice) with elements taken from the set ${\displaystyle \left\{1,\dots,g\right\}}$ of cardinality ${\displaystyle g}$ (number of possible values of each die), and the number of such multisets, i.e., ${\displaystyle w(n,g)}$ is the multiset coefficient

$$ {\displaystyle w(n,g)=\left\langle {\begin{matrix}g\\n\end{matrix}}\right\rangle ={g+n-1 \choose g-1}={g+n-1 \choose n}={\frac {(g+n-1)!}{n!(g-1)!}}} $$

which is exactly the same as the formula for ${\displaystyle w(n,g)}$, as derived above with the aid of a theorem involving binomial coefficients, namely

$$ {\displaystyle \sum _{k=0}^{n}{\frac {(k+a)!}{k!a!}}={\frac {(n+a+1)!}{n!(a+1)!}}.} $$

To understand the decomposition

$$ {\displaystyle w(n,g)=\sum _{k=0}^{n}w(n-k,g-1)=w(n,g-1)+w(n-1,g-1)+\dots +w(1,g-1)+w(0,g-1)} $$

or for example, ${\displaystyle n=4}$ and ${\displaystyle g=3}$ ${\displaystyle w(4,3)=w(4,2)+w(3,2)+w(2,2)+w(1,2)+w(0,2),}$

let us rearrange the elements of ${\displaystyle S(4,3)}$ as follows ${\displaystyle S(4,3)=\left\{\underbrace {(1111),(1112),(1122),(1222),(2222)} _{(\alpha )},\underbrace {(111{\color {Red}{\underset {=}{3}}}),(112{\color {Red}{\underset {=}{3}}}),(122{\color {Red}{\underset {=}{3}}}),(222{\color {Red}{\underset {=}{3}}})} _{(\beta )},\underbrace {(11{\color {Red}{\underset {==}{33}}}),(12{\color {Red}{\underset {==}{33}}}),(22{\color {Red}{\underset {==}{33}}})} _{(\gamma )},\underbrace {(1{\color {Red}{\underset {===}{333}}}),(2{\color {Red}{\underset {===}{333}}})} _{(\delta )}\underbrace {({\color {Red}{\underset {====}{3333}}})} _{(\omega )}\right\}.}$

Clearly, the subset ${\displaystyle (\alpha )}$ of ${\displaystyle S(4,3)}$ is the same as the set ${\displaystyle S(4,2)=\left\{(1111),(1112),(1122),(1222),(2222)\right\}.}$

By deleting the index ${\displaystyle m_{4}=3}$ (shown in red with double underline ) in the subset ${\displaystyle (\beta )}$ of ${\displaystyle S(4,3)}$, one obtains the set ${\displaystyle S(3,2)=\left\{(111),(112),(122),(222)\right\}.}$

In other words, there is a one-to-one correspondence between the subset ${\displaystyle (\beta )}$ of ${\displaystyle S(4,3)}$ and the set ${\displaystyle S(3,2)}$. We write ${\displaystyle (\beta )\longleftrightarrow S(3,2).}$

Similarly, it is easy to see that ${\displaystyle (\gamma )\longleftrightarrow S(2,2)=\left\{(11),(12),(22)\right\}}$ ${\displaystyle (\delta )\longleftrightarrow S(1,2)=\left\{(1),(2)\right\}}$ ${\displaystyle (\omega )\longleftrightarrow S(0,2)=\{\}=\varnothing.}$

Thus we can write ${\displaystyle S(4,3)=\bigcup _{k=0}^{4}S(4-k,2)}$ or more generally,

$$ {\displaystyle S(n,g)=\bigcup _{k=0}^{n}S(n-k,g-1);} $$

and since the sets ${\displaystyle S(i,g-1),{\text{ for }}i=0,\dots,n}$ are non-intersecting, we thus have

$$ {\displaystyle w(n,g)=\sum _{k=0}^{n}w(n-k,g-1),} $$

with the convention that

$$ {\displaystyle w(0,g)=1\,\forall g,{\text{ and }}w(n,0)=1\,\forall n.} $$

Continuing the process, we arrive at the following formula ${\displaystyle w(n,g)=\sum _{k_{1}=0}^{n}\sum _{k_{2}=0}^{n-k_{1}}w(n-k_{1}-k_{2},g-2)=\sum _{k_{1}=0}^{n}\sum _{k_{2}=0}^{n-k_{1}}\cdots \sum _{k_{g}=0}^{n-\sum _{j=1}^{g-1}k_{j}}w(n-\sum _{i=1}^{g}k_{i},0).}$ Using the convention (7) 2 above, we obtain the formula

$$ {\displaystyle w(n,g)=\sum _{k_{1}=0}^{n}\sum _{k_{2}=0}^{n-k_{1}}\cdots \sum _{k_{g}=0}^{n-\sum _{j=1}^{g-1}k_{j}}1,} $$

keeping in mind that for ${\displaystyle q}$ and ${\displaystyle p}$ being constants, we have

It can then be verified that (8) and (2) give the same result for ${\displaystyle w(4,3)}$, ${\displaystyle w(3,3)}$, ${\displaystyle w(3,2)}$, etc.

## Interdisciplinary applications

Viewed as a pure probability distribution, the Bose–Einstein distribution has found application in other fields:

- In recent years, Bose–Einstein statistics has also been used as a method for term weighting in information retrieval. The method is one of a collection of DFR ("Divergence From Randomness") models, the basic notion being that Bose–Einstein statistics may be a useful indicator in cases where a particular term and a particular document have a significant relationship that would not have occurred purely by chance. Source code for implementing this model is available from the Terrier project at the University of Glasgow.
- The evolution of many complex systems, including the World Wide Web, business, and citation networks, is encoded in the dynamic web describing the interactions between the system's constituents. Despite their irreversible and nonequilibrium nature these networks follow Bose statistics and can undergo Bose–Einstein condensation. Addressing the dynamical properties of these nonequilibrium systems within the framework of equilibrium quantum gases predicts that the "first-mover-advantage", "fit-get-rich" (FGR) and "winner-takes-all" phenomena observed in competitive systems are thermodynamically distinct phases of the underlying evolving networks.

In recent years, Bose–Einstein statistics has also been used as a method for term weighting in information retrieval. The method is one of a collection of DFR ("Divergence From Randomness") models, the basic notion being that Bose–Einstein statistics may be a useful indicator in cases where a particular term and a particular document have a significant relationship that would not have occurred purely by chance. Source code for implementing this model is available from the Terrier project at the University of Glasgow.

The evolution of many complex systems, including the World Wide Web, business, and citation networks, is encoded in the dynamic web describing the interactions between the system's constituents. Despite their irreversible and nonequilibrium nature these networks follow Bose statistics and can undergo Bose–Einstein condensation. Addressing the dynamical properties of these nonequilibrium systems within the framework of equilibrium quantum gases predicts that the "first-mover-advantage", "fit-get-rich" (FGR) and "winner-takes-all" phenomena observed in competitive systems are thermodynamically distinct phases of the underlying evolving networks.


------------------------------------------------------------------